
# GNSS Satellite Clock & Ephemeris Error Prediction
## SIH 2025 — Results Notebook

**Model:** Gaussian Process Regression  
**Evaluation metric:** Shapiro-Wilk W statistic on residuals  
**Benchmark:** W = 0.9810, p = 0.5840

---


In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from scipy import stats
from IPython.display import Image, display

# Paths (adjust if running from a different directory)
import os
BASE = os.path.abspath('..')  # project root
RES  = os.path.join(BASE, 'results')
FIG  = os.path.join(BASE, 'figures')
PROC = os.path.join(BASE, 'Data', 'Processed')

ERR_COLS = ['x_error','y_error','z_error','clock_error']
print("Setup complete.")



## 1. Dataset Summary

Three independent satellites, each with 7 days of training data.


In [ ]:
summary = {
    'Satellite' : ['GEO (A)','MEO1 (B)','MEO2 (C)'],
    'Train rows': [142, 46, 143],
    'Test rows' : [69, 6, 18],
    'Train start': ['2025-09-01','2025-09-01','2025-09-03'],
    'Test date'  : ['2025-09-08','2025-09-08','2025-09-10'],
    'Upload mode': ['120-min→15-min','Irregular','Irregular'],
    'Key challenge': ['Upload spikes ±58m',
                      'Only 46 rows','4 daily data gaps'],
}
df_sum = pd.DataFrame(summary)
print(df_sum.to_string(index=False))



## 2. Priority 1 — Shapiro-Wilk Results


In [ ]:
sw_report = pd.read_csv(os.path.join(RES, 'submission_sw_report.csv'))
avg = sw_report[sw_report['column'].isin(['AVERAGED','GRAND AVERAGE'])]
print("SW Scores (averaged over 4 error columns):")
print(avg[['satellite','column','sw_w','sw_p',
           'h0_rejected','rmse']].to_string(index=False))
print(f"\nBenchmark: W=0.9810  p=0.5840  H0_rejected=0")



## 3. Master Figure — All Results Summary


In [ ]:
display(Image(os.path.join(FIG, 'phase10_master_figure.png'), width=900))



## 4. Prediction Dashboard — All Satellites × All Columns


In [ ]:
display(Image(os.path.join(FIG, 'phase10_dashboard.png'), width=900))



## 5. Q-Q Plots (Priority 3)

Points on the diagonal = normal residuals = high SW_W score.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, (sat, fname) in enumerate([
    ('GEO',  'phase8_qq_geo.png'),
    ('MEO1', 'phase8_qq_meo1.png'),
    ('MEO2', 'phase8_qq_meo2.png'),
]):
    img = plt.imread(os.path.join(FIG, fname))
    axes[i].imshow(img)
    axes[i].axis('off')
    axes[i].set_title(sat, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()



## 6. GEO Spike Analysis — Explaining the Limitation

For smooth rows (|error| < 10m): **SW_W ≥ 0.95** for y_error and z_error.  
Upload-boundary spikes (±35–58m) are **operationally driven** — not predictable  
from orbital mechanics. This is a data limitation, not a modeling failure.


In [ ]:
display(Image(os.path.join(FIG, 'phase10_geo_spike_analysis.png'), width=900))



## 7. Residual Histograms — All Satellites


In [ ]:
display(Image(os.path.join(FIG, 'phase8_residual_hist_all.png'), width=900))



## 8. SW_W Progress Across All Phases


In [ ]:
display(Image(os.path.join(FIG, 'phase8_final_comparison.png'), width=700))



## 9. Priority 2 — Residual Mean and Standard Deviation


In [ ]:
p2 = sw_report[sw_report['column'].isin(['AVERAGED','GRAND AVERAGE'])]
print("Priority 2 — Residual Statistics:")
print(p2[['satellite','column','res_mean','res_std',
          'rmse','mae']].to_string(index=False))
print()
print("NOTE: MEO1 and MEO2 residual means ≈ 0 (no bias)")
print("      GEO  residual mean = +0.38m (caused by upload spikes)")



## 10. Conclusion

| Satellite | SW_W | All columns pass H0? | Key insight |
|---|---|---|---|
| GEO  | 0.7865 | ✗ | Upload spikes ±58m are not learnable |
| MEO1 | **0.9084** | **✓** | All 4 columns normal, near benchmark |
| MEO2 | 0.8076 | Partial | 24h data gaps cause extrapolation error |

**Why Gaussian Process:**
- Only 46–143 training rows → no room for LSTM windows
- Non-uniform sampling (1–1556 min gaps) → GP handles naturally
- Evaluation is residual normality → GP posterior is Gaussian by design

**Runtime:** < 60 seconds for all 3 satellites on any laptop.
